# Workbook 2 · ¿Aprobamos o rechazamos el lote? Clasificación con ML

**Taller: Toma de decisiones en Ingeniería con Machine Learning** · Práctica de extensión (para casa o sesión posterior)

En el Workbook 1 predijimos **un número** (resistencia, en MPa): eso es **regresión**. Aquí predeciremos **una categoría** (aprobado / rechazado): eso es **clasificación**. El flujo de trabajo es el mismo.

**Situación:** una planta recibe lotes de vino tinto y, antes de embotellar, mide 11 propiedades fisicoquímicas (acidez, azúcar, azufre, alcohol…). Una cata sensorial da la calificación final (0 a 10), pero catar cada lote es lento y caro. **¿Podemos usar los análisis de laboratorio para decidir qué lotes merecen aprobarse y cuáles deben revisarse?**

Este caso es análogo al control de calidad de cualquier proceso químico o de manufactura.

## 1 · Librerías y datos

**Datos:** *Wine Quality* (Cortez et al., 2009), UCI Machine Learning Repository, licencia CC BY 4.0. Usaremos el archivo de **vino tinto** (`winequality-red.csv`).

- Página: https://archive.ics.uci.edu/dataset/186/wine+quality
- DOI: 10.24432/C56S3T

In [ ]:
import io
import os
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

URL_VINO = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
RUTAS_LOCALES = ["datos/winequality-red.csv", "winequality-red.csv"]   # copia local opcional


def cargar_vino():
    for ruta in RUTAS_LOCALES:
        if os.path.exists(ruta):
            print("Leyendo copia local:", ruta)
            return pd.read_csv(ruta, sep=";")
    print("Descargando desde UCI...")
    with urllib.request.urlopen(URL_VINO, timeout=60) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    return pd.read_csv(z.open("winequality-red.csv"), sep=";")


df = cargar_vino()
df.columns = df.columns.str.replace(" ", "_")
print("Listo:", df.shape[0], "lotes y", df.shape[1], "columnas")
df.head()

> Si la descarga falla, el instructor te pasará `winequality-red.csv`: súbelo con el ícono de carpeta 📁 y vuelve a ejecutar la celda.

## 2 · Convertir el problema en una decisión

La calificación (`quality`) va de 0 a 10. Definimos una **regla de negocio** (ilustrativa): un lote se **aprueba** si su calificación es **6 o más**.

In [ ]:
print("Distribución de calificaciones:")
print(df["quality"].value_counts().sort_index())

df["aprobado"] = (df["quality"] >= 6).astype(int)      # 1 = aprobado, 0 = rechazado
print()
print("Proporción de lotes aprobados:", round(df["aprobado"].mean(), 2))

**Pregunta:** un modelo que **siempre diga "aprobado"** acertaría tantas veces como la proporción de aprobados que acabas de ver (por ejemplo, si el 60 % de los lotes se aprueba, acertaría el 60 % de las veces) **sin aprender nada**. Esa es la línea base que cualquier modelo debe superar. Anota la proporción de arriba.

## 3 · Explorar: ¿qué distingue a un lote aprobado?

In [ ]:
variables = ["alcohol", "volatile_acidity", "sulphates", "citric_acid"]
fig, ejes = plt.subplots(1, 4, figsize=(15, 4))
for eje, col in zip(ejes, variables):
    datos = [df.loc[df["aprobado"] == 0, col], df.loc[df["aprobado"] == 1, col]]
    eje.boxplot(datos)
    eje.set_xticks([1, 2])
    eje.set_xticklabels(["Rechazado", "Aprobado"])
    eje.set_title(col)
plt.tight_layout()
plt.show()

Una caja más alta para "Aprobado" indica que esa variable tiende a ser mayor en los lotes buenos. ¿Qué dos variables parecen separar mejor ambos grupos?

## 4 · Separar entrenamiento y prueba

Usamos `stratify=y` para que ambos grupos mantengan la **misma proporción** de aprobados y rechazados.

In [ ]:
X = df.drop(columns=["quality", "aprobado"])
y = df["aprobado"]

X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Entrenamiento:", len(X_ent), "· Prueba:", len(X_prueba))

## 5 · Tres modelos de clasificación

Los tres se usan igual: `.fit()` para aprender y `.predict()` para decidir.

- **Regresión logística:** estima la *probabilidad* de que un lote sea aprobado a partir de una combinación ponderada de variables.
- **Árbol de decisión:** una cascada de preguntas de sí/no. Se puede dibujar y **leer como reglas**.
- **Bosque aleatorio:** muchos árboles votando.

In [ ]:
modelos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Árbol (profundidad 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Bosque aleatorio": RandomForestClassifier(n_estimators=200, random_state=42),
}

filas = []
for nombre, modelo in modelos.items():
    modelo.fit(X_ent, y_ent)
    pred = modelo.predict(X_prueba)
    filas.append({
        "Modelo": nombre,
        "Exactitud": accuracy_score(y_prueba, pred),
        "Precisión": precision_score(y_prueba, pred),
        "Exhaustividad": recall_score(y_prueba, pred),
    })
tabla = pd.DataFrame(filas).round(2)
tabla

**Cómo leer las métricas** (la clase de interés es "aprobado"):

| Métrica | Pregunta que responde |
|---|---|
| **Exactitud** | De todos los lotes, ¿qué fracción clasificó bien? |
| **Precisión** | De los que el modelo **aprobó**, ¿qué fracción de verdad era buena? (alta = pocos falsos aprobados) |
| **Exhaustividad** | De los lotes que **sí eran buenos**, ¿qué fracción logró aprobar? (alta = pocos falsos rechazados) |

La **exactitud sola engaña** cuando las clases están desbalanceadas; por eso miramos precisión y exhaustividad.

## 6 · Un árbol que se puede leer

Los árboles son especiales porque **explican su decisión** en forma de reglas. Eso es valioso en ingeniería, donde hay que justificar por qué se rechazó un lote.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(modelos["Árbol (profundidad 3)"], feature_names=list(X.columns),
          class_names=["Rechazado", "Aprobado"], filled=True, rounded=True, fontsize=8)
plt.show()

Lee el árbol de arriba hacia abajo: en cada caja, si la condición es verdadera se va a la **izquierda**; si es falsa, a la **derecha**. ¿Qué variable usa primero? ¿Tiene sentido químico?

## 7 · La matriz de confusión del bosque aleatorio

In [ ]:
bosque = modelos["Bosque aleatorio"]
pred_bosque = bosque.predict(X_prueba)

cm = confusion_matrix(y_prueba, pred_bosque)
ConfusionMatrixDisplay(cm, display_labels=["Rechazado", "Aprobado"]).plot(colorbar=False)
plt.xlabel("Decisión del modelo")
plt.ylabel("Realidad")
plt.grid(False)
plt.show()

falsos_aprobados = cm[0, 1]
falsos_rechazados = cm[1, 0]
print("Falsos aprobados (lote malo que pasa):", falsos_aprobados)
print("Falsos rechazados (lote bueno que se retiene):", falsos_rechazados)

## 8 · Ajustar el nivel de exigencia: el umbral de decisión

Los modelos de clasificación no solo dicen "aprobado/rechazado": pueden dar la **probabilidad** de ser aprobado. Nosotros decidimos **desde qué probabilidad aprobamos**. Subir ese umbral hace al proceso más exigente.

In [ ]:
prob_aprobado = bosque.predict_proba(X_prueba)[:, 1]

filas = []
for umbral in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    decision = (prob_aprobado >= umbral).astype(int)
    cm_u = confusion_matrix(y_prueba, decision)
    filas.append({
        "Umbral": umbral,
        "Falsos aprobados": int(cm_u[0, 1]),
        "Falsos rechazados": int(cm_u[1, 0]),
        "Lotes aprobados": int(decision.sum()),
    })
pd.DataFrame(filas)

In [ ]:
tabla_u = pd.DataFrame(filas)
plt.plot(tabla_u["Umbral"], tabla_u["Falsos aprobados"], marker="o", label="Falsos aprobados (riesgo de calidad)")
plt.plot(tabla_u["Umbral"], tabla_u["Falsos rechazados"], marker="o", label="Falsos rechazados (desperdicio)")
plt.xlabel("Umbral de probabilidad para aprobar")
plt.ylabel("Número de lotes")
plt.title("El compromiso entre dos errores")
plt.legend()
plt.show()

**Decisión de ingeniería:** no existe un umbral "correcto" universal; depende de **cuánto cuesta cada error**. Si un lote malo que llega al cliente cuesta 10 veces más que retener uno bueno, conviene un umbral alto. Si retener es muy caro (producto perecedero), uno más bajo.

## 9 · ¿Qué variables importan?

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=X.columns).sort_values()
importancias.plot(kind="barh", color="#55A868")
plt.xlabel("Importancia relativa")
plt.title("Variables más usadas por el bosque aleatorio")
plt.show()

## 10 · Tu turno

1. Cambia la regla de aprobación a `quality >= 7` (más exigente). ¿Qué pasa con la proporción de aprobados y con las métricas? (Pista: se trata de un problema más **desbalanceado**.)
2. Cambia `max_depth=3` del árbol por `max_depth=10`. ¿Mejora en prueba? Compara con la exactitud en entrenamiento: ¿hay sobreajuste?
3. Elige el umbral de la sección 8 que, a tu criterio, mejor equilibra los dos errores si **un falso aprobado cuesta 5 veces un falso rechazado**. Justifica tu elección en 2 líneas.
4. Reflexión: ¿aprobarías automáticamente lotes sin cata usando este modelo? ¿Qué tendría que cumplirse antes?

## Conclusiones

- Regresión (predecir un número) y clasificación (predecir una categoría) comparten el mismo flujo: **datos → explorar → separar → entrenar → evaluar → decidir**.
- Con clases desbalanceadas, la **exactitud engaña**: revisa precisión y exhaustividad.
- El **umbral** convierte una probabilidad en una decisión y refleja el **costo de cada error**.
- Los árboles permiten **explicar** la decisión; los bosques suelen **acertar más** pero explican menos.

*Datos: Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Wine Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56S3T (CC BY 4.0).*

---
## Soluciones orientativas

In [ ]:
# Solución 1 · Regla más exigente (calidad >= 7)
y7 = (df["quality"] >= 7).astype(int)
print("Proporción de aprobados con calidad >= 7:", round(y7.mean(), 2))
Xe, Xp, ye, yp = train_test_split(X, y7, test_size=0.25, random_state=42, stratify=y7)
m7 = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xe, ye)
p7 = m7.predict(Xp)
print("Exactitud:", round(accuracy_score(yp, p7), 2),
      "· Precisión:", round(precision_score(yp, p7), 2),
      "· Exhaustividad:", round(recall_score(yp, p7), 2))
# Con pocos aprobados, la exactitud sube "sola" pero la exhaustividad cae: el modelo deja pasar pocos lotes buenos.

In [ ]:
# Solución 2 · Profundidad del árbol
for prof in [3, 10]:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=42).fit(X_ent, y_ent)
    print(f"Profundidad {prof}: exactitud entrenamiento = {arbol.score(X_ent, y_ent):.2f} · prueba = {arbol.score(X_prueba, y_prueba):.2f}")
# Un árbol profundo casi memoriza el entrenamiento, pero en prueba no mejora (o empeora): sobreajuste.

In [ ]:
# Solución 3 · Umbral con costo 5:1 (un falso aprobado cuesta 5 veces un falso rechazado)
tabla_u["Costo total"] = 5 * tabla_u["Falsos aprobados"] + 1 * tabla_u["Falsos rechazados"]
print(tabla_u[["Umbral", "Falsos aprobados", "Falsos rechazados", "Costo total"]].to_string(index=False))
mejor = tabla_u.loc[tabla_u["Costo total"].idxmin(), "Umbral"]
print("Umbral de menor costo total:", mejor)

**Solución 4 (orientativa):** no sin condiciones. Antes habría que validar el modelo con lotes nuevos de la propia planta (no solo con estos datos públicos), definir el costo real de cada error, mantener cata por muestreo para detectar deriva del proceso y reservar una zona intermedia de probabilidades (por ejemplo, 0.4 a 0.7) para revisión humana.